# 🏠 Kaggle Starter Notebook: House Prices - Advanced Regression Techniques

Welcome to your first Kaggle Regression Notebook! In this project, we analyze residential home features from Ames, Iowa to predict final sale prices (`SalePrice`).

### 🎯 Project Objectives
1. **Exploratory Data Analysis (EDA)**: Understand target distribution, skewness, and key correlation factors.
2. **Data Cleaning & Preprocessing**: Handle missing values, encode categorical variables, and apply log transformations.
3. **Feature Engineering**: Construct domain-specific features like total square footage and total bathroom count.
4. **Machine Learning Pipelines**: Compare Ridge Regression, Random Forest, and LightGBM using 5-Fold Cross Validation (RMSLE).
5. **Kaggle Submission Export**: Generate a formatted `submission.csv` ready for competition submission.

## 1. Setup & Environment Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error

# ML Models
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("LightGBM not detected locally. Will fall back to GradientBoostingRegressor.")
    from sklearn.ensemble import GradientBoostingRegressor

# Formatting & Styling
pd.set_option('display.max_columns', 100)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("✅ Setup complete. Libraries loaded successfully!")

## 2. Data Acquisition & Inspection

We attempt to load `train.csv` and `test.csv` from `./input/` or current directory. If the dataset files are not present locally, a synthetic sample dataset is automatically generated for local testing.

In [ ]:
def generate_synthetic_house_data(n_samples=500):
    """Generates synthetic house price data for local demonstration."""
    np.random.seed(42)
    gr_liv_area = np.random.randint(800, 3500, size=n_samples)
    overall_qual = np.random.randint(1, 10, size=n_samples)
    total_bsmt_sf = np.random.randint(0, 2000, size=n_samples)
    year_built = np.random.randint(1950, 2021, size=n_samples)
    neighborhood = np.random.choice(['CollgCr', 'Veenker', 'Crawfor', 'NoRidge', 'Mitchel'], size=n_samples)
    
    noise = np.random.normal(0, 15000, size=n_samples)
    sale_price = 30000 + (gr_liv_area * 60) + (overall_qual * 15000) + (total_bsmt_sf * 40) + ((year_built - 1950) * 500) + noise
    sale_price = np.maximum(sale_price, 50000)
    
    df = pd.DataFrame({
        'Id': np.arange(1, n_samples + 1),
        'MSSubClass': np.random.choice([20, 60, 70, 120], size=n_samples),
        'Neighborhood': neighborhood,
        'OverallQual': overall_qual,
        'YearBuilt': year_built,
        'TotalBsmtSF': total_bsmt_sf,
        'GrLivArea': gr_liv_area,
        'FullBath': np.random.randint(1, 4, size=n_samples),
        'HalfBath': np.random.randint(0, 2, size=n_samples),
        'GarageCars': np.random.randint(0, 4, size=n_samples),
        'SalePrice': sale_price
    })
    return df

# Check for standard Kaggle paths or local files
train_path = 'train.csv'
test_path = 'test.csv'

if os.path.exists('../input/house-prices-advanced-regression-techniques/train.csv'):
    train_df = pd.read_csv('../input/house-prices-advanced-regression-techniques/train.csv')
    test_df = pd.read_csv('../input/house-prices-advanced-regression-techniques/test.csv')
    print("Loaded Kaggle dataset from ../input/")
elif os.path.exists(train_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    print("Loaded dataset from local folder.")
else:
    print("⚠️ Local train.csv not found. Generating synthetic demo dataset...")
    raw_df = generate_synthetic_house_data(600)
    train_df = raw_df.iloc[:400].copy()
    test_df = raw_df.iloc[400:].copy().drop(columns=['SalePrice'])

print(f"Training set shape: {train_df.shape}")
print(f"Testing set shape:  {test_df.shape}")
display(train_df.head(5))

## 3. Exploratory Data Analysis (EDA)

In regression problems where target values (`SalePrice`) span wide dollar amounts, raw target distributions are often right-skewed. Taking the natural log transformation $\log(1 + x)$ normalizes variance and helps algorithms perform better under Root Mean Squared Logarithmic Error (RMSLE).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw Target Distribution
sns.histplot(train_df['SalePrice'], kde=True, ax=axes[0], color='royalblue')
axes[0].set_title("Raw SalePrice Distribution (Right-Skewed)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("SalePrice ($)")

# Log-Transformed Target Distribution
log_price = np.log1p(train_df['SalePrice'])
sns.histplot(log_price, kde=True, ax=axes[1], color='forestgreen')
axes[1].set_title("Log-Transformed log1p(SalePrice) (Bell Curve / Normal)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("log1p(SalePrice)")

plt.tight_layout()
plt.show()

### Feature Correlations with Target

In [ ]:
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
corrs = train_df[numeric_cols].corr()['SalePrice'].sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=corrs.values[1:8], y=corrs.index[1:8], palette='viridis')
plt.title("Top Numerical Features Correlated with SalePrice", fontsize=12, fontweight='bold')
plt.xlabel("Pearson Correlation Coefficient")
plt.show()

## 4. Feature Engineering & Preprocessing Pipeline

We create domain features:
- `TotalSF`: Combining basement, 1st floor, and 2nd floor square footage (if present).
- `TotalBath`: Combining full and half bathrooms.
- `HouseAge`: Age of property at sales time.

In [ ]:
def engineer_features(df):
    df = df.copy()
    
    # Total Square Footage
    bsmt = df['TotalBsmtSF'] if 'TotalBsmtSF' in df.columns else 0
    gr_liv = df['GrLivArea'] if 'GrLivArea' in df.columns else 0
    df['TotalSF'] = bsmt + gr_liv
    
    # Total Bathrooms
    full_bath = df['FullBath'] if 'FullBath' in df.columns else 0
    half_bath = df['HalfBath'] if 'HalfBath' in df.columns else 0
    df['TotalBath'] = full_bath + (0.5 * half_bath)
    
    # House Age
    if 'YearBuilt' in df.columns:
        df['HouseAge'] = 2026 - df['YearBuilt']
    
    return df

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)

# Target & Features Split
y_train_log = np.log1p(train_fe['SalePrice'])
X_train = train_fe.drop(columns=['Id', 'SalePrice'], errors='ignore')
X_test = test_fe.drop(columns=['Id', 'SalePrice'], errors='ignore')

# Identify column types
num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Preprocessing pipelines
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

print(f"Engineered Features Count: {X_train.shape[1]} (Numerical: {len(num_features)}, Categorical: {len(cat_features)})")

## 5. Model Training & 5-Fold Cross-Validation

We evaluate 3 popular regression models using 5-Fold Cross Validation. We record the **Root Mean Squared Logarithmic Error (RMSLE)**.

In [ ]:
# Candidate Models
models = {
    'Ridge Regression': Ridge(alpha=10.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

if HAS_LGBM:
    models['LightGBM'] = LGBMRegressor(n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)
else:
    models['Gradient Boosting'] = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, random_state=42)

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    full_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    # Negated MSE on log-transformed y gives RMSLE directly
    scores = cross_val_score(full_pipeline, X_train, y_train_log, cv=kf, scoring='neg_mean_squared_error')
    rmsle_scores = np.sqrt(-scores)
    results[name] = rmsle_scores
    print(f"📊 {name}: Mean RMSLE = {rmsle_scores.mean():.4f} (Std = {rmsle_scores.std():.4f})")

### Visualizing Cross-Validation Model Comparison

In [ ]:
results_df = pd.DataFrame(results)
plt.figure(figsize=(8, 4))
sns.boxplot(data=results_df, palette='Set2')
plt.title("5-Fold Cross-Validation Performance Comparison (Lower RMSLE is Better)", fontsize=12, fontweight='bold')
plt.ylabel("RMSLE")
plt.show()

## 6. Final Predictions & Kaggle Submission Generation

We select the top performing model, fit it on the full training dataset, predict on `X_test`, inverse log-transform predictions with `np.expm1()`, and save to `submission.csv`.

In [ ]:
# Select Best Model
best_model_name = min(results, key=lambda k: results[k].mean())
print(f"🏆 Winning Model selected: {best_model_name}")

best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', models[best_model_name])
])

# Fit on full training data
best_pipeline.fit(X_train, y_train_log)

# Predict on test set & inverse log transform
test_log_preds = best_pipeline.predict(X_test)
final_preds = np.expm1(test_log_preds)

# Build submission DataFrame
sub_df = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': final_preds
})

sub_df.to_csv('submission.csv', index=False)
print("💾 Successfully saved 'submission.csv'!")
display(sub_df.head(10))

## 7. Summary & Kaggle Submission Instructions

### 📊 Data Analysis Key Findings
- **Target Distribution**: `SalePrice` is positively right-skewed. Applying `np.log1p()` normalized target variance, improving model stability and alignment with Kaggle's RMSLE metric.
- **Key Predictors**: Overall Quality (`OverallQual`), Living Area (`GrLivArea`), Total Basement SF (`TotalBsmtSF`), and engineered total area (`TotalSF`) have the highest positive correlation with final house prices.
- **Model Performance**: Ensemble tree models (Random Forest / LightGBM) consistently outperform standard linear baselines by capturing non-linear interactions.

### 🚀 Next Steps & Further Improvements
- **Hyperparameter Tuning**: Use `Optuna` or `GridSearchCV` to fine-tune `learning_rate` and `max_depth` in LightGBM/XGBoost.
- **Advanced Feature Engineering**: Add neighborhood price density groupings and aggregate quality interaction metrics.
- **Model Ensembling**: Blend weighted predictions of Ridge, Random Forest, XGBoost, and CatBoost models.

---

### 📌 How to Upload & Submit on Kaggle
1. Go to the **[House Prices Kaggle Competition Page](https://www.kaggle.com/c/house-prices-advanced-regression-techniques)**.
2. Click **Code** $\to$ **New Notebook**.
3. Copy-paste these cells into your Kaggle Notebook (or click **File $\to$ Upload Notebook** and select this `.ipynb` file).
4. Click **Run All** $\to$ **Save Version** (Quick Save or Commit).
5. Under **Output**, find `submission.csv` and click **Submit to Competition**! 🎉